In [16]:
import pandas as pd
import json

In [17]:
df = pd.read_csv('./data/clean/ai_fat_wave3_long.csv',encoding='latin-1')

In [18]:
# create an association_number column tracking the association number
df['association_number'] = df.groupby('code').cumcount() + 1

df['association_code'] = df['code'] + '_' + df['association_number'].astype(str)

In [19]:
associations = {}
for association in df['association_correct']:
    if association not in associations:
        associations[association] = df[df['association_correct'] == association]['association_code'].tolist()

# save to json file with indentation
with open('./data/clean/associations.json', 'w') as f:
    json.dump(associations, f, indent=4)

In [20]:
# create a list of only keys
association_keys = list(associations.keys())
with open('./data/clean/associations_keys.json', 'w') as f:
    json.dump(association_keys, f, indent=4)

In [21]:
df.to_csv('./data/clean/ai_fat_wave3_long.csv', index=False, encoding='utf-8')

In [26]:
df_long = pd.read_csv('./data/clean/ai_fat_wave3_long.csv', encoding='utf-8')
df_wide = pd.read_csv('./data/clean/ai_fat_wave3.csv', encoding='utf-8')


pivot = df_long.pivot_table(
    index='code',
    columns='association_number',
    values='association_correct',
    aggfunc='first'
)

pivot.columns = [f'association_correct{int(c)}' for c in pivot.columns]
wide = df_wide.merge(pivot, on='code', how='left')

# replace NAs with empty strings
for col in [
    "association_correct1",
    "association_correct2",
    "association_correct3",
    "association_correct4",
    "association_correct5"
]:
    if col in wide.columns:
        wide[col] = wide[col].fillna('')

def replace_nas(row):
    d = {
        "association1": row["association_correct1"],
        "association2": row["association_correct2"],
        "association3": row["association_correct3"],
        "association4": row["association_correct4"],
        "association5": row["association_correct5"],
    }
    j = json.dumps(d)           
    j = j.replace('"', "'") # keep single quotes
    return j

wide['associations_correct'] = wide.apply(replace_nas, axis=1)


corrected_cols = list(df_wide.columns) + [
    "association_correct1",
    "association_correct2",
    "association_correct3",
    "association_correct4",
    "association_correct5",
    "associations_correct"
]

wide = wide[corrected_cols]


wide.to_csv('./data/clean/ai_fat_wave3_corr.csv',
            index=False,
            encoding='utf-8')